# Treasury auction cycle (EDGE-E1)

Does the auctioned tenor cheapen against its par-curve neighbours in the 5 business days before a nominal coupon auction and richen in the 3 days after? Pre-registration `research/prereg/E1.yaml`; fitting period 2015 to 2020, holdout 2021-01-01 to 2026-06-30; data: Treasury par curve, TreasuryDirect auction records, Federal Reserve GSW curve (fixtures in `data/fixtures/edge/`, manifests `data/manifests/edge_*.json`). Daily closes: nothing here says anything about intraday edge.

Every number below is recomputed from the committed fixtures and checked against the ledger row; the plain-language finding is in `docs/research/EDGE_STUDIES.md`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch.edge import report

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent


def show(df: pd.DataFrame, digits: int = 4) -> None:
    display(Markdown(ba.to_markdown(df.round(digits))))

In [ ]:
from ratesresearch.edge import e1_auction

res = e1_auction.run(ROOT)
row = report.ledger_row(ROOT, "EDGE-E1")
display(Markdown(report.check_against_ledger(res, row)))
show(report.tests_table(res, row))

Band points and periods (unadjusted p; only the central holdout row is a registered test).

In [ ]:
show(report.band_table(res, "pre"))

In [ ]:
show(report.band_table(res, "post"))

In [ ]:
show(pd.DataFrame(res["band"]["central"]["holdout_by_tenor"]).T.reset_index(names="tenor"))

In [ ]:
g = res["band"]["central"]["gsw_holdout"]
show(
    pd.DataFrame(
        [
            {"measure": k, **{a: g[k][a] for a in ("n", "mean", "ci_low", "ci_high")}}
            for k in ("pre", "post")
        ]
    )
)